# T1 blood INR — Colab 启动器

**这个 notebook 不含任何算法代码。** 它只做四件事：拉代码、装依赖、挂 Drive、启动。
算法只有一份，在 GitHub 仓库的 `train_inr_unsup_spiral.py` 里 —— 本地和 Colab 跑的是**同一个 commit**。

之前的做法是把训练代码内联进 notebook，那样两边会悄悄分叉，
正是 `AGENTS.md` 里 matched-arms 规则要防的事。


## 1. 拉代码（记下 commit，这就是 provenance）

In [ ]:
!git clone -q https://github.com/ArtieXu/t1-blood-inr.git
%cd t1-blood-inr
!git log -1 --format='commit %H%n日期   %ad'

## 2. 依赖

tiny-cuda-nn 要现编，约 5–10 分钟。

In [ ]:
!pip install -q torchkbnufft==1.5.2 ninja imageio h5py scikit-image
!pip install -q git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch
!python tools/deps.py

## 3. 挂 Drive 取数据

数据永远不进仓库。

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/T1_blood_INR_v2/02_data_reference/gassp1_data.mat'
OUT  = '/content/drive/MyDrive/T1_blood_INR_v2/results'
import os; assert os.path.exists(DATA), DATA; os.makedirs(OUT, exist_ok=True)

## 4. 环境 + 启动

`expandable_segments:False` 是 WSL 上排查出来的（见 `docs/BLOCKERS_20260819.md`）。
Colab 是原生 Linux，未必需要 —— 但设了无害，而且保证两边算子行为一致。

`--ckpt_every` 是 Colab 的**必需品**：会话随时可能断，断了用同一条命令续跑。

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:False'
os.environ['WANDB_MODE'] = 'disabled'

EXP    = 'u0_1600'
EPOCHS = 1600
RUNDIR = f'{OUT}/{EXP}'
os.makedirs(RUNDIR, exist_ok=True)

import glob
prev = sorted(glob.glob(f'{RUNDIR}/log/*/'))
resume = ['--resume', prev[-1]] if prev and os.path.exists(prev[-1] + 'ckpt.pt') else []
print('续跑自' if resume else '全新开始', resume[-1] if resume else '')

%cd {RUNDIR}
!python -u /content/t1-blood-inr/train_inr_unsup_spiral.py \
    --gpu 0 --epochs {EPOCHS} --seed 0 --kb_grid_size 324 \
    --summary_epoch 100 --ckpt_every 50 --tag {EXP} \
    --data_path {DATA} --dc_form feng_rel --dc_weighting uniform \
    {' '.join(resume)}
%cd /content/t1-blood-inr

## 5. 验收

会话断了就从第 4 格重跑，它会自动接上断点。

In [ ]:
import glob
d = sorted(glob.glob(f'{RUNDIR}/log/*/'))[-1]
!python tools/check_run.py {d} --expect_epochs {EPOCHS}